<a href="https://colab.research.google.com/github/Adewol3/Fly-rank-internship/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adewol3/Fly-rank-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Refresh / Content Opportunity Scoring

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**What decision does the work improve?**

Which pages a content or SEO team reviews first, out of a much larger pool than they have time to check by hand. Right now that decision gets made by gut feel, whoever's page catches attention, or a simple rule like "sort by traffic drop." The work improves the ordering of that review queue, not the decision to review at all. That distinction matters: you're not deciding whether to act, you're deciding what order to act in, given fixed capacity.



**Who acts on it?**

A human reviewer, likely a content strategist or SEO analyst at one of FlyRank's client accounts. Important detail: it's never the model making the final call. The model produces a ranked list with reason codes, the human decides whether to actually refresh, expand, prune, or leave a page alone. This is stated directly in the guide (section 6): "decision-support," not automation. Naming this person concretely, not "the business," is what makes your framing paragraph read like you understand the actual workflow instead of describing an abstract pipeline.

**What does a wrong recommendation cost, and is it symmetric?**

It's not symmetric, and that's the part worth being explicit about:



*   **False positive (flagged as worth reviewing, wasn't actually a problem):** the reviewer spends 20-30 minutes checking a page that didn't need it. Annoying, low cost, recoverable.
*   **False negative (a genuinely declining page never makes the list):** the page keeps losing traffic silently until someone notices some other way, or doesn't. For a client paying for this service, that's lost visibility and, eventually, lost trust in the tool. Higher cost, harder to recover.





In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
from google.colab import files
uploaded = files.upload()

Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv


In [4]:
import pandas as pd
df = pd.read_csv("content_refresh_anonymized.csv")

In [5]:
# Step 1: confirm what you have
print(df.shape)
df.dtypes

(30000, 44)


,0
content_id,object
client_id,object
search_volume,float64
competition,float64
competition_level,object
cpc,float64
content_type,object
main_intent,object
word_count,float64
char_count,float64


In [6]:
# Step 2: apply the same filter the starter pipeline uses
# (matches docs/ml-intern-dataset-and-lane-guide.md section 5)
filtered = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
filtered = filtered.drop_duplicates(subset="content_id")

print("Raw rows:", len(df))
print("Eligible rows after filter:", len(filtered))

Raw rows: 30000
Eligible rows after filter: 30000


In [7]:
# Step 3: the numbers that argue for Lane 2 (Refresh/Content Opportunity Scoring)

# how many pages are declining under the starter label
declining = filtered["trend_direction"] == "down"
print("Declining share:", round(declining.mean(), 3))
print("Declining count:", declining.sum())

# of those declining, how many have real demand behind them
# (this is the guide's 'declining_with_demand' reason code logic)
declining_with_demand = filtered[declining & (filtered["impressions_90d"] >= 100)]
print("Declining pages with impressions_90d >= 100:", len(declining_with_demand))
print("Share of ALL eligible pages this represents:", round(len(declining_with_demand) / len(filtered), 3))

Declining share: 0.542
Declining count: 16262
Declining pages with impressions_90d >= 100: 13152
Share of ALL eligible pages this represents: 0.438


In [8]:
# Step 4: sanity check the numbers against a couple of related columns,
# so you're not reading meaning into a fluke
print(filtered["impressions_90d"].describe())
print(filtered["trend_direction"].value_counts())

count     30000.000000
mean       5200.366300
std       16838.019547
min           1.000000
25%          81.000000
50%         731.000000
75%        3615.250000
max      517715.000000
Name: impressions_90d, dtype: float64
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*


**What this work can say:**

- *Observed*: which pages, under the filters and windows I defined, showed declining trend alongside real search demand (impressions_90d >= 100). These are measurements, not interpretations.
- *Directional*: that a learned ranking (random forest, per the starter pipeline) surfaces more true decliners in its top 50 than a fixed rule does, on this dataset, under this validation split. "This suggests a ranked model is worth building" — not "this proves it will work everywhere."
- *Decision-support*: a ranked queue with reason codes, meant to help a reviewer spend limited time on the most promising pages first. The output changes what a human looks at, not what happens to the page. The refresh decision, and any edit made, is still theirs.

**What this work will never say:**

- *Causal proof*: I cannot claim a refresh caused a recovery. Correlation between "page was flagged" and "page later improved" isn't evidence of causation, that requires an actual experiment (e.g., A/B test on refreshed vs. untouched matched pages), which this dataset doesn't give me.
- *Predicting Google*: I'm not modeling Google's ranking algorithm. I'm modeling FlyRank's own observed search and engagement signals (impressions, clicks, position, sessions) against a label I defined myself. Any pattern I find is a pattern in this data, not a discovered ranking factor.
- *Precision as certainty*: even a high-confidence label (per the guide's threshold: top 20% score, minimum volume, model probability >= 0.50) is still a probability, not a guarantee. A flagged page might genuinely not need action. The reviewer's judgment stays load-bearing.

**Why the line matters:**

This isn't a disclaimer tacked on for compliance. It's the actual boundary of what supervised learning on observational data can prove. Claiming more than this would mean either overselling the tool to whoever reads the capstone paper, or fooling myself into thinking I've solved something I haven't. The guide is explicit about this in section 14 (public-safe output rules): "we observed," "this suggests," never "this proves."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.